In [1]:
import numpy as np
from scipy.optimize import curve_fit

In [2]:
def arrhenius_fit(T_vect, k, a_guess):    
    R = 1.987  # [cal/mol/K]
    # Define model function for 3-parameter fit
    def modelfun(T, a1, a2, a3):
        return a1 + a2*np.log(T) - a3/(R*T)
    # Fit the model
    popt, _ = curve_fit(modelfun, T_vect, np.log(k), p0=a_guess)
    # Extract parameters
    k0 = np.exp(popt[0])  # [cm3/mol/s]
    alfa = popt[1]        # [-]
    EA = popt[2]          # [cal/mol]
    # Calculate model predictions
    k_model = k0 * T_vect**alfa * np.exp(-EA/(R*T_vect))
    # Calculate quality metrics
    SSres_log = np.sum((np.log(k) - np.log(k_model))**2)
    SSres = np.sum((k - k_model)**2)
    SStot = np.sum((k - np.mean(k))**2)
    R2 = 1 - SSres/SStot
    # Calculate R2_log (adjusted R-squared for log fit)
    n = len(k)
    p = 3  # number of parameters
    R2_log = 1 - (1 - R2) * (n - 1)/(n - p - 1)
    
    return k0, alfa, EA, R2, R2_log, SSres_log

def plog_refit(P, A, alfa, Eact):
    # Define temperature range (300-2500K, similar to the original script)
    T = np.linspace(300, 2500, 100)
    # Calculate [M] concentration at each temperature (mol/cm3)
    # Using ideal gas law: [M] = P/(R*T) where R = 82.05746 cm3·atm/(mol·K)
    R_gas = 82.05746  # cm3·atm/(mol·K)
    M_conc = P / (R_gas * T)  # [mol/cm3]
    # Calculate PLOG rate k = k0[M] using original Arrhenius parameters
    R = 1.987  # cal/mol/K
    k_plog = A * (T**alfa) * np.exp(-Eact/(R*T))  # [cm6/mol2/s]
    # Calculate rate constant k0 = k/[M] (divide by M concentration)
    k0_values = k_plog / M_conc  # [cm3/mol/s]
    # Initial guess for refitting
    a_guess = [np.log(A), alfa, Eact]
    # Perform 3-parameter fit on the temperature-dependent k0 values
    k0_new, alfa_new, EA_new, R2, R2_log, SSres_log = arrhenius_fit(T, k0_values, a_guess)
    return k0_new, alfa_new, EA_new, R2

In [3]:
# Direct input parameters (modify these values as needed)
P = 0.1     # Pressure [atm]
A = 3.442e+17  # Pre-exponential factor [cm6/mol2/s]
alfa = -2.748 # Temperature exponent [-]
Eact = -750.78 # Activation energy [cal/mol]
    
# Perform refitting
k0, alfa_new, EA_new, R2 = plog_refit(P, A, alfa, Eact)
    
# Print results
print(f"\nPLOG Rate Refitting Results at P = {P} atm:")
print(f"==========================================")
print(f"Original parameters (PLOG rate):")
print(f"A = {A:.4e} [cm6/mol2/s]")
print(f"alfa = {alfa:.4f}")
print(f"Eact = {Eact:.2f} [cal/mol]")
print(f"\nRefitted parameters (k0):")
print(f"k0 = {k0:.4e} [cm3/mol/s]")
print(f"alfa = {alfa_new:.4f}")
print(f"Eact = {EA_new:.2f} [cal/mol]")
print(f"R² = {R2:.6f}")
    
# Print concentration of M and verification at multiple temperatures
R_gas = 82.05746  # cm3·atm/(mol·K)
print("\nVerification at selected temperatures:")
print(" Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]")
print("-----------------|----------------|----------------|--------------------")
    
for temp in [300, 1000, 2000]:
    M_conc = P / (R_gas * temp)  # [mol/cm3]
    k0_at_T = k0 * (temp**alfa_new) * np.exp(-EA_new/(1.987*temp))
    k_plog_at_T = A * (temp**alfa) * np.exp(-Eact/(1.987*temp))
    print(f"{temp:15.1f}  | {M_conc:14.4e} | {k0_at_T:14.4e} | {k0_at_T * M_conc:16.4e}")
        
print(f"\nOriginal PLOG at 300K: {A * (300**alfa) * np.exp(-Eact/(1.987*300)):.4e} [cm6/mol2/s]")


PLOG Rate Refitting Results at P = 0.1 atm:
Original parameters (PLOG rate):
A = 3.4420e+17 [cm6/mol2/s]
alfa = -2.7480
Eact = -750.78 [cal/mol]

Refitted parameters (k0):
k0 = 2.8244e+20 [cm3/mol/s]
alfa = -1.7480
Eact = -750.78 [cal/mol]
R² = 1.000000

Verification at selected temperatures:
 Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]
-----------------|----------------|----------------|--------------------
          300.0  |     4.0622e-06 |     4.6549e+16 |       1.8909e+11
         1000.0  |     1.2187e-06 |     2.3498e+15 |       2.8636e+09
         2000.0  |     6.0933e-07 |     5.7913e+14 |       3.5288e+08

Original PLOG at 300K: 1.8909e+11 [cm6/mol2/s]


In [4]:
import jax.numpy as jnp
from diffPLOG2TROE.rate_constants import Arrhenius, refit_arrhenius
from diffPLOG2TROE.physical_constants import constants

In [5]:
def plog_refit(P, plog):
    T = jnp.linspace(300, 2500, 100)
    conc = P / (constants.R_L_atm_K_mol * T) * jnp.float64(0.001)
    k_plog = plog.kinetic_constant(T)

    # Calculate rate constant k0 = k / [M] (divide by M concentration)
    k0_values = k_plog / conc  # [cm3/mol/s]

    # Perform 3-parameter fit on the temperature-dependent k0 values
    lnA_new, n_new, EaR_new = refit_arrhenius(k0_values, T, True)
    return jnp.exp(lnA_new), n_new, EaR_new * constants.R_cal_mol

In [6]:
P = 0.1
original_constant = Arrhenius(name="PLOG @ 0.1 atm", params=jnp.array([3.442e+17, -2.748, -750.78]))

# Perform refitting
A_new, n_new, EA_new = plog_refit(P, original_constant)
refitted_constant = Arrhenius(name="k0 refitted", params=jnp.array([A_new, n_new, EA_new]))
refitted_constant = Arrhenius(params=jnp.array([A_new, n_new, EA_new]))

# Print results
print(f"\nPLOG Rate Refitting Results at P = {P} atm:")
print("==========================================")
print(original_constant)
print(refitted_constant)

print("\nVerification at selected temperatures:")
print(" Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]")
print("-----------------|----------------|----------------|--------------------")
for T in [300, 1000, 2000]:
    conc = P / (constants.R_L_atm_K_mol * T) * jnp.float64(0.001)
    k0_at_T = refitted_constant.kinetic_constant(T)
    k_plog_at_T = original_constant.kinetic_constant(T)
    print(f"{T:15.1f}  | {conc:14.4e} | {k0_at_T:14.4e} | {k0_at_T * M_conc:16.4e}")


PLOG Rate Refitting Results at P = 0.1 atm:
PLOG @ 0.1 atm		3.44200e+17 -2.74800e+00 -7.50780e+02
unknown :(		2.82441e+20 -1.74800e+00 -7.50780e+02

Verification at selected temperatures:
 Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]
-----------------|----------------|----------------|--------------------
          300.0  |     4.0622e-06 |     4.6543e+16 |       2.8360e+10
         1000.0  |     1.2187e-06 |     2.3497e+15 |       1.4317e+09
         2000.0  |     6.0933e-07 |     5.7912e+14 |       3.5287e+08
